# SRQ-FLY CIFAR-100 D5 — locked train-only selection
This notebook never opens `test.pt`. It selects the missing CIFAR Ridge lambda on nested training splits, then evaluates the locked train-validation controls. Return the ZIP before any held-out run.

In [ ]:
# === Edit paths only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'paper/srq-fly-draft'
WORK_DIR = '/content/SOHO-CL'
DRIVE_FEATURE_CACHE = '/content/drive/MyDrive/T-SOHO/tsoho_cifar100_cache'
LOCAL_TRAIN_CACHE = '/content/srq_cifar_d5_train_cache'
LARGE_WTA_CACHE = '/content/drive/MyDrive/T-SOHO/srq_cifar_d5_wta_m10000_seed2025'
MATCHED_WTA_CACHE = '/content/drive/MyDrive/T-SOHO/srq_cifar_d5_wta_m4409_seed2025'
OUTPUT_DIR = '/content/drive/MyDrive/T-SOHO/srq_cifar_d5_outputs'
EXPECTED_CONFIG_SHA256 = '9af36b234d980962e6834a9ccd9f5204c9c7f660e44a5faaf9c6332e3151ea81'

In [ ]:
# Clone the exact branch and verify the locked config.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, os, shutil, subprocess, sys
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'], check=True)
config_path = Path('configs/srq_fly_cifar100_d5_train_only.json')
observed = hashlib.sha256(config_path.read_bytes()).hexdigest()
assert observed == EXPECTED_CONFIG_SHA256, (observed, EXPECTED_CONFIG_SHA256)
print('repo commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('locked config SHA-256:', observed)

In [ ]:
# Restore ONLY train.pt + metadata.json. Deliberately do not copy test.pt.
import json, torch
source = Path(DRIVE_FEATURE_CACHE)
target = Path(LOCAL_TRAIN_CACHE)
assert (source/'metadata.json').is_file() and (source/'train.pt').is_file(), 'Drive CIFAR cache missing'
if target.exists(): shutil.rmtree(target)
target.mkdir(parents=True)
for name in ('metadata.json','train.pt'): shutil.copy2(source/name, target/name)
assert not (target/'test.pt').exists(), 'FAIL: held-out test cache became visible'
metadata = json.loads((target/'metadata.json').read_text())
train = torch.load(target/'train.pt', weights_only=True, map_location='cpu')
assert metadata['dataset'] == 'CIFAR-100'
assert metadata['checkpoint_sha256'] == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
assert tuple(train['features'].shape) == (50000,768) and tuple(train['labels'].shape) == (50000,)
assert torch.isfinite(train['features']).all() and len(torch.unique(train['labels'])) == 100
print('TRAIN CACHE PASS:', tuple(train['features'].shape), '| test.pt absent')
del train

In [ ]:
# Correctness tests; synthetic data only.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_srq_fly_cifar_selection.py','tests/test_srq_fly_math.py','tests/test_srq_fly_learner.py'], check=True)
print('D5 correctness gate: PASS')

In [ ]:
# Locked train-only run. WTA CACHE shows encoding progress; INNER/OUTER TASK shows solves.
command = [sys.executable,'-u','tools/srq_fly_cifar_selection.py',
  '--config','configs/srq_fly_cifar100_d5_train_only.json',
  '--feature-cache-dir',LOCAL_TRAIN_CACHE,
  '--large-code-cache-dir',LARGE_WTA_CACHE,
  '--matched-code-cache-dir',MATCHED_WTA_CACHE,
  '--output-dir',OUTPUT_DIR,'--device','cuda']
print('Starting/resuming CIFAR D5. Five INNER candidates, then paired OUTER controls.', flush=True)
completed = subprocess.run(command)
assert completed.returncode == 0, 'D5 failed; return the complete traceback without editing config.'
assert Path(OUTPUT_DIR,'d5_results.json').is_file()
print('CIFAR D5 process: COMPLETE')

In [ ]:
# Compact summary and evidence download. WTA caches are intentionally excluded from the ZIP.
import pandas as pd
payload = json.loads(Path(OUTPUT_DIR,'d5_results.json').read_text())
rows=[]
for key in ('exact_fly_10000','srq_fly_10000','exact_fly_4409','raw_ridge'):
    item=payload[key]; rows.append({'method':key,'val_AA':item['validation_average_accuracy'],'final':item['stage_accuracy'][-1],'state_bytes':item['persistent_state_bytes'],'max_residual':item['maximum_solver_relative_residual']})
display(pd.DataFrame(rows).sort_values('val_AA', ascending=False))
print('status:', payload['status'])
print('selected lambda:', payload['selected_fly_and_srq_lambda'])
print('gates:', json.dumps(payload['gates'], indent=2))
archive = '/content/srq_fly_cifar_d5_train_only.zip'
shutil.make_archive(archive[:-4], 'zip', OUTPUT_DIR)
from google.colab import files
files.download(archive)